# ONNM - Colab training

Runs the **same code** as a local run: this notebook unpacks the repo, installs it, and
calls the same `scripts/` entry points. A result produced here is comparable to one
produced on the local RX 7900 XT, which is the whole point - the job is to explain a
regression, and that needs two runs that differ in exactly one thing.

## What this run is for

The overnight run scored **0.8629** macro ROC-AUC against the full run's **0.8905**.
ROC-AUC is threshold-independent, so that is a real loss of ranking quality, not something
a threshold can recover. Two things changed at once - aggressive augmentation and OHEM -
so neither can be blamed yet. Cells 9 and 10 run them separately.

## Before you start

1. `Runtime -> Change runtime type -> T4 GPU` (or better, if you have Pro).
2. This notebook expects the following in your Drive at
   **`MyDrive/OSTEONEURALNETWORK/`**:

   | file | what it is |
   |---|---|
   | `onnm-code.zip` | the repo source |
   | `BTXRD.zip` | the 874 MB dataset |
   | `splits.json` | the exact split used locally, for comparability |

3. Run the cells in order. Cell 8 is a 2-epoch smoke run - **do not skip it**, it proves
   the whole path works before an hour is spent on a real run.

**Free-tier caveat:** Colab disconnects on idle and caps sessions at ~12 h. The configs
below run 40 epochs with early stopping (patience 15), which lands well inside that. Do
not try to run `overnight.yaml`'s nominal 150 epochs here.


In [ ]:
# --- Cell 1: what hardware did we actually get? -----------------------------
# Worth knowing before anything else. The free tier is a T4 (Turing, sm_75),
# which has NO bfloat16 - the project trains in bf16 locally, and the training
# loop needs a GradScaler on fp16 that bf16 does not use. resolve_amp_dtype
# handles the fallback, but seeing it here means no surprises an hour in.
import subprocess

import torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout)
print("torch          ", torch.__version__)
print("cuda available ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device         ", torch.cuda.get_device_name(0))
    print("capability     ", torch.cuda.get_device_capability(0))
    print("bf16 supported ", torch.cuda.is_bf16_supported(), "  <- False on a T4; expected")
else:
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU")


In [ ]:
# --- Cell 2: mount Drive and check the three inputs are there ---------------
import os
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/OSTEONEURALNETWORK")
REPO = Path("/content/OsteoNeuralNetwork-Model")
DATA = Path("/content/data")

expected = ["onnm-code.zip", "BTXRD.zip", "splits.json"]
missing = [f for f in expected if not (DRIVE / f).exists()]
if missing:
    raise SystemExit(
        f"missing from {DRIVE}: {missing}\n"
        f"present: {sorted(p.name for p in DRIVE.iterdir()) if DRIVE.exists() else 'folder not found'}"
    )

for f in expected:
    size = (DRIVE / f).stat().st_size / 1024 ** 2
    print(f"  {f:16s} {size:9.1f} MB")


In [ ]:
# --- Cell 3: unpack code and data to local disk -----------------------------
# Unzipping onto Drive itself would be pathologically slow: BTXRD is ~3.7k small
# files and Drive is a network filesystem. Everything goes to /content, which is
# local SSD, and results are copied back at the end.
#
# The dataset is symlinked into the repo's default location rather than
# configured elsewhere, because verify_data.py and make_splits.py take no
# --profile flag and would otherwise look somewhere different from training.
import shutil

if not REPO.exists():
    !unzip -q "{DRIVE}/onnm-code.zip" -d /content/
print("code   ", REPO, "->", len(list(REPO.rglob("*.py"))), "python files")

if not (DATA / "BTXRD").exists():
    DATA.mkdir(parents=True, exist_ok=True)
    !unzip -q "{DRIVE}/BTXRD.zip" -d "{DATA}"
n_images = len(list((DATA / "BTXRD" / "images").glob("*")))
print("images ", DATA / "BTXRD", "->", n_images, "files   (expect 3746)")

# Symlink into the repo's default data_root, and copy the split in.
(REPO / "data" / "raw").mkdir(parents=True, exist_ok=True)
(REPO / "data" / "interim").mkdir(parents=True, exist_ok=True)
link = REPO / "data" / "raw" / "BTXRD"
if not link.exists():
    os.symlink(DATA / "BTXRD", link)
shutil.copy(DRIVE / "splits.json", REPO / "data" / "interim" / "splits.json")

# The split is copied rather than regenerated so Colab and local runs are
# literally the same partition. Comparing scores across different splits would
# be meaningless, and make_splits.py reproducing the same seed is not the same
# guarantee as using the same file.
import json
split = json.load(open(REPO / "data/interim/splits.json"))
print("split   train/val/test =",
      len(split["train"]), "/", len(split["val"]), "/", len(split["test"]))
print("        content_hash =", split["content_hash"], " (local run: db908a9afdc5d085)")
assert split["content_hash"] == "db908a9afdc5d085", "split differs from the local run"


In [ ]:
# --- Cell 4: install the project layer only ---------------------------------
# Colab ships a working CUDA torch. Do NOT reinstall it: replacing it reliably
# breaks the preinstalled CUDA libraries and costs a runtime restart.
#
# --no-deps on the editable install is what enforces that. pyproject.toml does
# not list torch (deliberately), but several of its dependencies do, and pip
# resolving them would happily pull a CPU-only wheel over Colab's CUDA build.
# Everything else the project needs is either installed on the line above or
# already present in Colab. Cell 5 (verify_env) is what confirms that claim
# rather than assuming it - if --no-deps skipped something real, it fails there.
%cd /content/OsteoNeuralNetwork-Model
!pip install -q monai==1.5.2 pydicom openpyxl
!pip install -q -e . --no-deps
print("installed")


## Gates

The project has numbered gates that must pass before a result means anything. Gate 6
(`overfit_check`) has **never been run** against the current pipeline - it proves gradients
actually reach the backbone. Running it here clears a blocking item from `TODO.md`.


In [ ]:
# --- Cell 5: gates 1 and 2 - environment, then data -------------------------
!python scripts/verify_env.py
!python scripts/verify_data.py


In [ ]:
# --- Cell 6: gate 3 - the test suite ----------------------------------------
# Needs no dataset; catches an install that half-worked.
!pip install -q pytest
!python -m pytest -q


In [ ]:
# --- Cell 7: gate 6 - overfit a tiny batch ----------------------------------
# A model that cannot drive loss to ~0 on 32 stratified images has a broken
# gradient path, and every number produced after that is noise. ~2 minutes.
!python scripts/overfit_check.py --profile colab --samples 32 --steps 200


In [ ]:
# --- Cell 8: smoke run - 2 epochs, end to end -------------------------------
# Proves loaders, AMP, scheduler, checkpointing and metrics all work together on
# this machine before an hour is committed. Look for two lines in the output:
#   'AMP: float16 (GradScaler on)'   - the T4 fallback fired correctly
#   'cudnn_enabled: True'            - the ROCm miopen=false flag was ignored
!python scripts/train.py \
    --override configs/densenet121_3class.yaml \
    --profile colab --epochs 2 --tag colab-smoke


## The two ablations

Each runs `overnight.yaml` with exactly one of its two changes removed. Together with the
existing `full` and `overnight` runs, that is enough to attribute the regression.

Budget roughly **45-70 min each** on a T4 at 40 epochs with early stopping. Run them one at
a time and keep the tab alive.


In [ ]:
# --- Cell 9: ablation A - OHEM alone ----------------------------------------
# Aggressive augmentation reverted to full_run strength; OHEM left on.
!python scripts/train.py \
    --override configs/densenet121_3class.yaml \
    --override configs/overnight.yaml \
    --override configs/ablations/ohem_only.yaml \
    --profile colab --epochs 40 --tag abl-ohem


In [ ]:
# --- Cell 10: ablation B - aggressive augmentation alone --------------------
# Augmentation as in overnight.yaml; OHEM disabled.
!python scripts/train.py \
    --override configs/densenet121_3class.yaml \
    --override configs/overnight.yaml \
    --override configs/ablations/augs_only.yaml \
    --profile colab --epochs 40 --tag abl-augs


In [ ]:
# --- Cell 11: calibrate and evaluate each run -------------------------------
# Threshold and temperature are fitted on VAL and applied unchanged to TEST.
# Never fit them on test (invariant 1) - scripts/calibrate.py warns if you try.
import glob

runs = sorted(glob.glob("reports/abl-*/best.pt"))
print("found:", runs)

for ckpt in runs:
    print("\n" + "=" * 70, "\n", ckpt, "\n" + "=" * 70)
    !python scripts/calibrate.py --checkpoint {ckpt} --profile colab --sweep
    !python scripts/evaluate.py  --checkpoint {ckpt} --profile colab --split test


In [ ]:
# --- Cell 12: side-by-side comparison ---------------------------------------
# The number that settles the question is val macro ROC-AUC, because it is
# threshold-independent: full 0.8905 vs overnight 0.8629. Whichever ablation
# lands near 0.86 is carrying the regression.
import json
import glob

rows = []
for hist_path in sorted(glob.glob("reports/*/history.json")):
    run = hist_path.split("/")[-2]
    history = json.load(open(hist_path))
    if not history:
        continue
    best = max(history, key=lambda e: e.get("val_roc_auc_macro", -1))
    rows.append({
        "run": run,
        "epochs": len(history),
        "val_roc_auc_macro": round(best.get("val_roc_auc_macro", float("nan")), 4),
        "val_malignant_recall": round(best.get("val_malignant_recall", float("nan")), 3),
        "val_normal_called_lesion": best.get("val_normal_called_lesion"),
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows).sort_values("val_roc_auc_macro", ascending=False))
except Exception:
    for r in rows:
        print(r)

print("\nreference (local runs):  full = 0.8905    overnight = 0.8629")


In [ ]:
# --- Cell 13: save results back to Drive ------------------------------------
# /content is wiped on disconnect. Copy the whole run directory, not just the
# checkpoint: history.json and metrics_*.json are what the comparison needs.
import shutil
from pathlib import Path

out = DRIVE / "reports"
out.mkdir(parents=True, exist_ok=True)

for run in sorted(Path("reports").glob("*")):
    if not run.is_dir():
        continue
    target = out / run.name
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(run, target)
    print("saved", target)

print("\nDone. These persist in Drive after the runtime disconnects.")


## Licence

BTXRD is **CC BY-NC-ND 4.0**. Keeping a private copy in your own Drive is not
redistribution and is fine. What is not fine:

- publishing the dataset or any part of it
- publishing **Grad-CAM overlays** - they are derived images, which the NoDerivatives
  clause covers

`data/` and `reports/` are gitignored for this reason. Keep it that way.

## What to do with the result

Whichever ablation lands near **0.86** is the one carrying the regression:

- **`abl-ohem` low** -> OHEM is the cause. Lower `loss.ohem.penalty` below 4.0 or raise
  `warmup_epochs` above 5, and note that malignant recall fell 0.653 -> 0.469 while false
  positives fell 65 -> 37, which is a bias shift rather than better discrimination.
- **`abl-augs` low** -> the augmentation is too aggressive. `dropout_prob: 0.5` with up to
  6 holes of 40 px on a 256 px image can occlude a lesion outright, which teaches the model
  that a lesion-free-looking film is still labelled lesion.
- **Both near 0.89** -> the two interact, and the combination is the problem rather than
  either part.
